In [ ]:
import pandas as pd
import datetime
import re
import csv

In [ ]:
df = pd.read_csv("sales_saleitens_filial_01.csv", sep=";", encoding="utf-8-sig", low_memory=False)

In [ ]:
df.columns.tolist()

In [ ]:
df.head(5)

In [ ]:
print(df["item"].unique())

In [ ]:
df_item = df['item'].str.replace(r'\d+,\d+\s*|\b\d{1,3}\b', '', regex=True)

df_limpo = df_item.unique()

In [ ]:
item_df = pd.DataFrame(df_limpo, columns=["item"])

In [ ]:
item_df

In [ ]:
valores_unicos = list(set(item_df["item"]))

with open("itens_limpos.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["item"])  # Define o cabeçalho da coluna
    for item in valores_unicos:
        writer.writerow([item])

In [ ]:
df_list = df_item.tolist()

In [ ]:
print(df_list)

In [ ]:
import pandas as pd

df = pd.read_csv(r"c:\Users\wtba2\pipeline\data\final\clean\receivables_filial_01.csv", sep=";", nrows=0)
print([c for c in df.columns if "receiv" in c.lower()])

In [ ]:
df = pd.read_csv(r"c:\Users\wtba2\pipeline\data\raw\receivables_filial_01_raw.csv", sep=";", nrows=0)
print([c for c in df.columns if "receiv" in c.lower()])

In [ ]:
#df["saledate"].dtype

In [ ]:
#df_sales_time = df["saledate"].iloc[0]
#print(df_sales_time)

In [ ]:
# Converter a coluna para datetime (caso ainda não esteja)
'''df["saledate_dt"] = pd.to_datetime(df["saledate"], errors="coerce")

# Extrair o ano
df["ano"] = df["saledate_dt"].dt.year

# Contar registros por ano
contagem_por_ano = df["ano"].value_counts().sort_index()

print(contagem_por_ano)'''

In [ ]:
df_stats = df["salevalue"].describe()

In [ ]:
print(df_stats)

In [ ]:
df["salevalue"].median()

In [3]:
import pandas as pd

RAW_DIR   = r"C:\Users\wtba2\pipeline\data\raw"
CLEAN_DIR = r"C:\Users\wtba2\pipeline\data\final\clean"

def fix_encoding(value):
    if not isinstance(value, str):
        return value
    try:
        return value.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return value

def normalize_columns(df):
    df.columns = [col.replace(" ", "_").replace(".", "_").lower() for col in df.columns]
    return df

def tratar_csv(
    caminho: str,
    colunas_drop: list[str] = [],
    id_col: str = None,
    sep: str = ";",
    encoding: str = "utf-8-sig",
) -> pd.DataFrame:
    df = pd.read_csv(caminho, sep=sep, encoding=encoding)
    print(f"Carregado: {df.shape[0]} linhas x {df.shape[1]} colunas")

    df = normalize_columns(df)
    df = df.apply(lambda col: col.map(fix_encoding) if col.dtype == object else col)

    if colunas_drop:
        presentes = [c for c in colunas_drop if c in df.columns]
        df = df.drop(columns=presentes, errors="ignore")
        print(f"Colunas removidas: {presentes}")

    if id_col and id_col in df.columns:
        antes = len(df)
        df = df.drop_duplicates(subset=[id_col])
        print(f"Duplicatas removidas: {antes - len(df)}")
    else:
        print(f"id_col '{id_col}' não encontrado — deduplicação ignorada")

    print(f"Resultado: {df.shape[0]} linhas x {df.shape[1]} colunas")
    return df


# ── Caminhos prontos ───────────────────────────────────────────────────────────

MEMBERS_RAW   = rf"{RAW_DIR}\members_filial_01_raw.csv"
MEMBERS_CLEAN = rf"{CLEAN_DIR}\members_filial_01.csv"

# Uso — raw:
df_raw = tratar_csv(
    caminho=MEMBERS_RAW,
    colunas_drop=["address", "neighborhood", "accesscardnumber", "membershipstatus",
                  "memberships", "idemployeepersonaltrainer", "nameemployeepersonaltrainer",
                  "idmembermigration", "responsibles", "personaltype", "cref",
                  "crefexpirationdate", "useridgurupass"],
    id_col="idmember",
)

# Uso — clean (já tratado, só carregar):
df_clean = pd.read_csv(MEMBERS_CLEAN, sep=";", encoding="utf-8-sig")

C:\Users\wtba2\AppData\Local\Temp\ipykernel_34148\1623802274.py:25: DtypeWarning: Columns (9,10,12,20,38,45,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(caminho, sep=sep, encoding=encoding)


Carregado: 18525 linhas x 51 colunas
Colunas removidas: ['address', 'neighborhood', 'accesscardnumber', 'membershipstatus', 'memberships', 'idemployeepersonaltrainer', 'nameemployeepersonaltrainer', 'idmembermigration', 'responsibles', 'personaltype', 'cref', 'crefexpirationdate', 'useridgurupass']
Duplicatas removidas: 0
Resultado: 18525 linhas x 38 colunas


C:\Users\wtba2\AppData\Local\Temp\ipykernel_34148\1623802274.py:63: DtypeWarning: Columns (9,10,12,20,38,45,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df_clean = pd.read_csv(MEMBERS_CLEAN, sep=";", encoding="utf-8-sig")


In [4]:
df_raw.to_csv(rf"{CLEAN_DIR}\members_filial_01_tratado.csv", sep=";", encoding="utf-8-sig", index=False)

df_raw.to_csv(MEMBERS_CLEAN, sep=";", encoding="utf-8-sig", index=False)